# Stacking ensembles

_Classical ML_

**Let several models vote, then train a model to combine the votes.**

Stacking blends several different models into one stronger model.

     First, many base models each make a prediction.

---

This notebook is a practice scaffold for the **Stacking ensembles** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

X, y = make_classification(n_samples=600, n_features=20, n_informative=8,
                           random_state=0)

base = [("rf", RandomForestClassifier(n_estimators=50, random_state=0)),
        ("knn", KNeighborsClassifier(n_neighbors=7))]
stack = StackingClassifier(estimators=base,
                           final_estimator=LogisticRegression(max_iter=1000),
                           cv=5)

for name, est in base + [("stack", stack)]:
    acc = cross_val_score(est, X, y, cv=5).mean()
    print("%-6s 5-fold accuracy = %.3f" % (name, acc))

## Visualize the data & results

_On the breast-cancer scan data, does stacking beat each base model alone?_

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True)   # 569 real tumor scans, 30 features
rf = RandomForestClassifier(n_estimators=100, random_state=0)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
stack = StackingClassifier(estimators=[("rf", rf), ("knn", knn)],
                           final_estimator=LogisticRegression(max_iter=1000), cv=5)

labels = ["RandomForest", "kNN", "Stacked"]
accs = [cross_val_score(m, X, y, cv=5).mean() for m in (rf, knn, stack)]

colors = ["#4ea1ff", "#4ea1ff", "#7ee787"]
plt.bar(labels, accs, color=colors)
for i, a in enumerate(accs):
    plt.text(i, a, "%.3f" % a, ha="center", va="bottom")
plt.ylim(0.94, 0.98)
plt.title("5-fold accuracy on breast cancer: base models vs stacked ensemble")
plt.ylabel("accuracy")
plt.show()